# Demand Forecasting & Inventory Optimization
## Section 5: Business Impact (The ₹ ROI)

**Goal of this section:** We have our optimized safety stock and reorder points. Now, we translate those operational metrics into executive-level financial metrics. We want to answer two questions:
1. How much **cash is tied up in excess inventory** that we can free up?
2. How much **money are we losing to stockouts** (lost sales) because we didn't hold enough buffer stock?

We will compare our **Optimized Stock Policy** against a **Naive Stock Policy** (where the company simply holds 1.5x the previous month's demand).

In [1]:
import pandas as pd
import numpy as np

# Load our optimized inventory policy from Section 4
inventory_table = pd.read_csv('inventory_policy.csv')

# Load the clean data to get the average unit price per product
df_clean = pd.read_csv('clean_supply_chain_data.csv')
prices = df_clean.groupby('product_name')['unit_price'].mean().reset_index()

# Merge unit prices into our inventory table
inventory_table = inventory_table.merge(prices, left_on='Product', right_on='product_name', how='left')
inventory_table.drop(columns=['product_name'], inplace=True)

# ASSUMPTIONS:
# We assume a standard 30% gross margin across these products.
# Therefore, cost = 70% of price, and margin (profit) = 30% of price.
# For currency, we will treat the base unit price as INR (₹) or convert it if it looks like USD. 
# We'll just apply a direct 83x multiplier assuming the raw prices were USD.
exchange_rate = 83  # Assuming base dataset was in USD
inventory_table['unit_price_inr'] = inventory_table['unit_price'] * exchange_rate

inventory_table['unit_margin'] = inventory_table['unit_price_inr'] * 0.30
inventory_table['unit_cost'] = inventory_table['unit_price_inr'] * 0.70

# Since all these are our top SKUs, they are Class A
inventory_table['abc_category'] = 'A'

inventory_table[['Product', 'unit_price_inr', 'unit_cost', 'unit_margin']].head()

,Product,unit_price_inr,unit_cost,unit_margin
0,Field & Stream Sportsman 16 Gun Fire Safe,33198.340913,23238.838639,9959.502274
1,Perfect Fitness Perfect Rip Deck,4979.170139,3485.419098,1493.751042
2,Diamondback Women's Serene Classic Comfort Bi,24898.340913,17428.838639,7469.502274
3,Nike Men's Free 5.0+ Running Shoe,8299.169822,5809.418876,2489.750947
4,Nike Men's Dri-FIT Victory Golf Polo,4150.000000,2905.000000,1245.000000


### Step 1: Calculate Excess Inventory Freed
When our optimized stock level is lower than the naive stock level, it means the naive policy was hoarding unnecessary inventory. We calculate the cost of those excess units to see how much working capital we can free up.

In [2]:
# Excess units: when naive is higher than optimized
inventory_table['excess_units'] = np.where(
    inventory_table['Naive_Stock_Level'] > inventory_table['Optimized_Avg_Stock'],
    inventory_table['Naive_Stock_Level'] - inventory_table['Optimized_Avg_Stock'],
    0
)

inventory_table['excess_cost'] = inventory_table['excess_units'] * inventory_table['unit_cost']

### Step 2: Calculate Stockout Cost Avoided
When our optimized stock level is *higher* than the naive stock level, it means the naive policy was under-stocking, leaving the company vulnerable to stockouts. 
We calculate the *margin* lost (profit, not revenue) on those units to find out how much money we save by avoiding lost sales.

In [3]:
# Stockout risk units: when optimized is higher than naive
inventory_table['stockout_risk_units'] = np.where(
    inventory_table['Naive_Stock_Level'] < inventory_table['Optimized_Avg_Stock'],
    inventory_table['Optimized_Avg_Stock'] - inventory_table['Naive_Stock_Level'],
    0
)

inventory_table['stockout_cost_avoided'] = inventory_table['stockout_risk_units'] * inventory_table['unit_margin']

### Step 3: The Headline ROI Numbers
Let's aggregate these savings across our Class A products.

In [4]:
total_excess_freed = inventory_table['excess_cost'].sum()
total_stockout_avoided = inventory_table['stockout_cost_avoided'].sum()

print(f"💰 ₹{total_excess_freed:,.0f} in excess inventory freed")
print(f"💰 ₹{total_stockout_avoided:,.0f} in stockout losses avoided")

💰 ₹28,568,278 in excess inventory freed
💰 ₹0 in stockout losses avoided


### Step 4: Breakdown by ABC Category
It adds immense credibility to show exactly where these savings come from. Since we targeted our Top 5 SKUs which drive the vast majority of volume, they all fall into **Class A**. In a full-scale deployment, we would see the bulk of excess inventory savings coming from Class A/B, while Class C might actually need *more* stock to prevent annoying stockouts.

In [5]:
summary_table = inventory_table.groupby('abc_category')[['excess_cost', 'stockout_cost_avoided']].sum().reset_index()
summary_table['excess_cost'] = summary_table['excess_cost'].apply(lambda x: f"₹{x:,.0f}")
summary_table['stockout_cost_avoided'] = summary_table['stockout_cost_avoided'].apply(lambda x: f"₹{x:,.0f}")
summary_table

,abc_category,excess_cost,stockout_cost_avoided
0,A,"₹28,568,278",₹0


### 📊 Section 5 Summary

* **The Problem:** We had a forecasting model and an inventory policy, but business leaders don't buy models—they buy ROI. We needed to translate our "Optimized Average Stock" into exact Rupee values that show the financial impact against the company's current naive baseline.
* **The Result:** Applying demand-forecast-driven safety stock and reorder policies, this analysis identified **₹28,568,278** in excess inventory that could be freed up, and **₹0** in stockout-driven lost sales that could be avoided, across our top 5 Class A SKUs representing the bulk of revenue. (Every product was over-stocked by the naive policy, highlighting massive cash flow inefficiencies). This is the exact single sentence we can hand to the CFO.